# ML Assignment — Healthcare EHR Classification
Binary classification with severe class imbalance (~95% label=0, ~5% label=1). Data is pre-scaled.

In [ ]:
# STEP 1 — Load all splits
import pandas as pd
import numpy as np
import joblib
import os
os.makedirs("models", exist_ok=True)

X_train_d1 = pd.read_pickle("data/processed/X_train_d1.pkl")
X_test_d1  = pd.read_pickle("data/processed/X_test_d1.pkl")
y_train_d1 = pd.read_pickle("data/processed/y_train_d1.pkl")
y_test_d1  = pd.read_pickle("data/processed/y_test_d1.pkl")

X_train_d2 = pd.read_pickle("data/processed/X_train_d2.pkl")
X_test_d2  = pd.read_pickle("data/processed/X_test_d2.pkl")
y_train_d2 = pd.read_pickle("data/processed/y_train_d2.pkl")
y_test_d2  = pd.read_pickle("data/processed/y_test_d2.pkl")

feature_names = joblib.load("data/processed/feature_names.pkl")

print("=== Shapes ===")
print(f"X_train_d1: {X_train_d1.shape}, X_test_d1: {X_test_d1.shape}")
print(f"X_train_d2: {X_train_d2.shape}, X_test_d2: {X_test_d2.shape}")
print("\n=== Class distributions ===")
print("y_train_d1:", pd.Series(y_train_d1).value_counts().to_dict())
print("y_test_d1: ", pd.Series(y_test_d1).value_counts().to_dict())
print("y_train_d2:", pd.Series(y_train_d2).value_counts().to_dict())
print("y_test_d2: ", pd.Series(y_test_d2).value_counts().to_dict())
print(f"\nFeature count: {len(feature_names)}")

In [ ]:
# STEP 2 — Apply SMOTE to both D1 and D2 training sets
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_d1_smote, y_train_d1_smote = smote.fit_resample(X_train_d1, y_train_d1)

smote2 = SMOTE(random_state=42)
X_train_d2_smote, y_train_d2_smote = smote2.fit_resample(X_train_d2, y_train_d2)

print("D1 after SMOTE:", pd.Series(y_train_d1_smote).value_counts().to_dict())
print("D2 after SMOTE:", pd.Series(y_train_d2_smote).value_counts().to_dict())

In [ ]:
# STEP 3 — Define evaluation function
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              ConfusionMatrixDisplay, RocCurveDisplay)

def evaluate(model, X, y, label=""):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:,1]
    return {
        "model": label,
        "accuracy": accuracy_score(y, preds),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
        "f1": f1_score(y, preds, zero_division=0),
        "roc_auc": roc_auc_score(y, probs)
    }

print("evaluate() defined.")

In [ ]:
# STEP 4 — Train Decision Tree with GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

dt_params = {"max_depth": [3, 5, 10, None]}
dt_grid = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=42),
    dt_params, scoring="f1", cv=5, n_jobs=-1
)
dt_grid.fit(X_train_d1, y_train_d1)
dt_best = dt_grid.best_estimator_
print("Best DT params:", dt_grid.best_params_)
joblib.dump(dt_best, "models/dt_best.pkl")

In [ ]:
# STEP 5 — Train SVM with GridSearchCV
from sklearn.svm import SVC

svm_params = {"C": [0.1, 1, 10]}
svm_grid = GridSearchCV(
    SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42),
    svm_params, scoring="f1", cv=5, n_jobs=-1
)
svm_grid.fit(X_train_d1, y_train_d1)
svm_best = svm_grid.best_estimator_
print("Best SVM params:", svm_grid.best_params_)
joblib.dump(svm_best, "models/svm_best.pkl")

In [ ]:
# STEP 6 — Train MLP on SMOTE-resampled D1
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    max_iter=500,
    early_stopping=True,
    random_state=42
)
mlp.fit(X_train_d1_smote, y_train_d1_smote)
joblib.dump(mlp, "models/mlp_d1.pkl")
print("MLP trained, iterations:", mlp.n_iter_)

In [ ]:
# STEP 7 — Evaluate all 3 models on BOTH test sets (6 combinations)
results = []
for model, name in [(dt_best, "DT"), (svm_best, "SVM"), (mlp, "MLP")]:
    r1 = evaluate(model, X_test_d1, y_test_d1, label=name)
    r1["evaluated_on"] = "D1"
    r2 = evaluate(model, X_test_d2, y_test_d2, label=name)
    r2["evaluated_on"] = "D2"
    results.extend([r1, r2])

metrics_df = pd.DataFrame(results)
metrics_df.to_csv("models/metrics_baseline.csv", index=False)
print(metrics_df.to_string())

In [ ]:
# STEP 8 — Plot ROC curves (2 subplots: D1 test and D2 test)
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors = {"DT": "blue", "SVM": "orange", "MLP": "green"}

for ax, (X_test, y_test, title) in zip(axes, [
    (X_test_d1, y_test_d1, "D1 Test (Historical)"),
    (X_test_d2, y_test_d2, "D2 Test (Current)")
]):
    for model, name in [(dt_best, "DT"), (svm_best, "SVM"), (mlp, "MLP")]:
        probs = model.predict_proba(X_test)[:,1]
        fpr, tpr, _ = roc_curve(y_test, probs)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.2f})", color=colors[name])
    ax.plot([0,1],[0,1],"k--")
    ax.set_title(f"ROC Curves — {title}")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend()

plt.tight_layout()
plt.savefig("models/roc_curves.png", dpi=150)
plt.show()
plt.close()
print("Saved: models/roc_curves.png")

In [ ]:
# STEP 9 — Plot confusion matrices (2x3 grid)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
models_list = [(dt_best, "Decision Tree"), (svm_best, "SVM"), (mlp, "MLP")]

for col, (model, name) in enumerate(models_list):
    for row, (X_test, y_test, ds_name) in enumerate([
        (X_test_d1, y_test_d1, "D1"),
        (X_test_d2, y_test_d2, "D2")
    ]):
        cm = confusion_matrix(y_test, model.predict(X_test))
        disp = ConfusionMatrixDisplay(cm)
        disp.plot(ax=axes[row][col], colorbar=False)
        axes[row][col].set_title(f"{name} on {ds_name}")

plt.suptitle("Confusion Matrices — All Models x Both Datasets", fontsize=14)
plt.tight_layout()
plt.savefig("models/confusion_matrices.png", dpi=150)
plt.show()
plt.close()
print("Saved: models/confusion_matrices.png")

In [ ]:
# STEP 10 — Feature importance plot (Decision Tree)
importances = dt_best.feature_importances_
feat_imp = pd.Series(importances, index=feature_names).sort_values(ascending=False)
top20 = feat_imp.head(20)

plt.figure(figsize=(10, 8))
top20.sort_values().plot(kind="barh", color="steelblue")
plt.title("Top 20 Feature Importances — Decision Tree")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig("models/feature_importance_dt.png", dpi=150)
plt.show()
plt.close()
print("Saved: models/feature_importance_dt.png")

In [ ]:
# STEP 11 — Continual Learning via partial_fit on D2
mlp_cl = joblib.load("models/mlp_d1.pkl")
batch_size = 64
classes = np.array([0, 1])
n_epochs = 50

X_d2_arr = X_train_d2_smote.values if hasattr(X_train_d2_smote, "values") else X_train_d2_smote
y_d2_arr = y_train_d2_smote.values if hasattr(y_train_d2_smote, "values") else y_train_d2_smote

for epoch in range(n_epochs):
    indices = np.random.permutation(len(X_d2_arr))
    for start in range(0, len(X_d2_arr), batch_size):
        batch_idx = indices[start:start+batch_size]
        mlp_cl.partial_fit(X_d2_arr[batch_idx], y_d2_arr[batch_idx], classes=classes)

joblib.dump(mlp_cl, "models/mlp_continual.pkl")

# Compare MLP before and after CL on D2 test
r_before = evaluate(mlp, X_test_d2, y_test_d2, label="MLP_D1")
r_before["evaluated_on"] = "D2"
r_after = evaluate(mlp_cl, X_test_d2, y_test_d2, label="MLP_CL")
r_after["evaluated_on"] = "D2"

cl_df = pd.DataFrame([r_before, r_after])
cl_df.to_csv("models/metrics_continual.csv", index=False)
print(cl_df.to_string())

In [ ]:
# STEP 12 — Continual learning comparison bar chart
metrics_to_plot = ["accuracy", "precision", "recall", "f1", "roc_auc"]
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, cl_df[cl_df.model=="MLP_D1"][metrics_to_plot].values[0],
               width, label="MLP_D1 (before CL)", color="steelblue")
bars2 = ax.bar(x + width/2, cl_df[cl_df.model=="MLP_CL"][metrics_to_plot].values[0],
               width, label="MLP_CL (after CL)", color="darkorange")

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1)
ax.set_title("Continual Learning Impact — MLP Before vs After Fine-tuning on D2")
ax.set_ylabel("Score")
ax.legend()
plt.tight_layout()
plt.savefig("models/continual_learning_comparison.png", dpi=150)
plt.show()
plt.close()
print("Saved: models/continual_learning_comparison.png")

In [ ]:
# STEP 13 — Final summary
print("\n=== BASELINE METRICS ===")
print(metrics_df.to_string(index=False))
print("\n=== CONTINUAL LEARNING METRICS ===")
print(cl_df.to_string(index=False))
print("\nAll files saved to models/:")
for f in sorted(os.listdir("models/")):
    print(f" - {f}")